# Deployment Workshop: From Training to Production

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pmcray/Prometheus_v0_PoC/blob/main/notebooks/deployment_workshop.ipynb)

**Goal**: Learn how to deploy your trained agents to play online on real game servers.

**Time**: ~60 minutes

**What You'll Learn**:
- How to deploy Go bots to OGS (Online Go Server)
- How to deploy Chess bots to Lichess
- Bot account setup and API authentication
- Managing multiple bots
- Monitoring, debugging, and maintenance
- Scaling for high-traffic scenarios

**Prerequisites**: Trained Go or Chess agent (or use our examples)

## Setup

If running on Google Colab, install Prometheus first:

In [ ]:
# Uncomment if running on Colab
# !git clone https://github.com/pmcray/Prometheus_v0_PoC.git
# %cd Prometheus_v0_PoC
# !pip install -q -e .

In [ ]:
import sys
import os
import json
import time
from pathlib import Path
from datetime import datetime

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

from prometheus.models.go_models import PrometheusGoAgent
from prometheus.models.chess_models import PrometheusChessAgent
from prometheus.configs import ModelBuilder

print("✓ Imports successful")
print(f"TensorFlow version: {tf.__version__}")

---

## Part 1: Introduction to Bot Deployment

### What is Bot Deployment?

**Deployment** means:
- Taking your trained agent
- Connecting it to an online game server
- Letting it play games against humans and other bots
- Running 24/7 (optionally)

### Why Deploy?

**Benefits**:
- ✅ **Real-world testing**: See how your agent performs against diverse opponents
- ✅ **Continuous learning**: Collect game data for further training
- ✅ **Community engagement**: Let others play against your bot
- ✅ **Benchmarking**: Get official ratings (ELO, rank)
- ✅ **Validation**: Prove your agent works in production

### Deployment Options

| Platform | Game | Free | Official Rating | Difficulty |
|----------|------|------|----------------|------------|
| **OGS** | Go | ✅ Yes | ✅ Yes | ⭐ Easy |
| **Lichess** | Chess | ✅ Yes | ✅ Yes | ⭐ Easy |
| **KGS** | Go | ✅ Yes | ✅ Yes | ⭐⭐ Medium |
| **Chess.com** | Chess | ❌ No | ✅ Yes | ⭐⭐⭐ Hard |

**We'll focus on OGS (Go) and Lichess (Chess)** - easiest to get started!

---

## Part 2: Deploying to OGS (Online Go Server)

### Step 1: Create a Bot Account

**Process**:

1. **Go to OGS**: https://online-go.com
2. **Create account**: Sign up normally (email + password)
3. **Request bot status**: Contact moderators via forum or email
   - Forum: https://forums.online-go.com
   - Include: Bot name, brief description, your main account
4. **Wait for approval**: Usually 1-2 days
5. **Get API token**: Profile → Settings → API Access

**Bot Naming Convention**:
- Use "Bot" or "AI" suffix: `PrometheusBot`, `MyGoAI`
- Be descriptive: `MCTSBot9x9`, `NeuralGoBot`
- Avoid impersonating humans

### Step 2: Set Up Authentication

**Store your API token securely**:

In [ ]:
# Create .env file for API credentials
env_content = """
# OGS Bot Configuration
OGS_USERNAME=YourBotName
OGS_API_TOKEN=your_api_token_here

# Lichess Bot Configuration (we'll add this later)
LICHESS_TOKEN=your_lichess_token_here
"""

# Save to file
env_path = Path('.env')
if not env_path.exists():
    with open(env_path, 'w') as f:
        f.write(env_content.strip())
    print("✓ Created .env file")
    print("\n⚠️  IMPORTANT: Edit .env and add your actual API tokens!")
else:
    print("✓ .env file already exists")

print("\n💡 Never commit .env to git! (Already in .gitignore)")

### Step 3: Prepare Your Agent

Let's create or load a trained agent for deployment.

In [ ]:
print("Preparing Go agent for deployment...\n")

# Option 1: Load existing model
model_path = Path('models/go_9x9.h5')

if model_path.exists():
    print(f"Loading existing model: {model_path}")
    agent = PrometheusGoAgent(board_size=9)
    agent.model = tf.keras.models.load_model(str(model_path))
    print(f"✓ Loaded agent with {agent.model.count_params():,} parameters")
else:
    print("No existing model found. Creating new agent...")
    # Option 2: Create new agent (you'll need to train this)
    agent = (
        ModelBuilder()
        .go(board_size=9)
        .strength('medium')
        .prometheus()
        .build()
    )
    print(f"✓ Created new agent with {agent.model.count_params():,} parameters")
    print("\n⚠️  WARNING: This agent is untrained! Train it first before deploying.")

# Add MCTS for stronger play
print("\nEnhancing with MCTS...")
from prometheus.mcts import add_mcts
agent = add_mcts(agent, num_simulations=400, preset='standard')
print("✓ MCTS enabled (400 simulations per move)")
print("  Expected strength: +300-500 ELO")

### Step 4: Test Locally

Before deploying, test that your agent can make valid moves.

In [ ]:
print("Testing agent locally...\n")

from prometheus.environments.go import GoEnvironment

# Create test environment
env = GoEnvironment(board_size=9)
state = env.reset()

# Play a few moves
for move_num in range(5):
    # Get agent's move
    move = agent.get_move(state)
    
    # Check if valid
    legal_moves = env.get_legal_moves()
    
    if move in legal_moves or move == env.board_size**2:  # Pass is always legal
        print(f"Move {move_num + 1}: {move} ✓ (Legal)")
    else:
        print(f"Move {move_num + 1}: {move} ✗ (ILLEGAL!)")
        print("  ⚠️  Agent is making illegal moves! Fix before deploying.")
        break
    
    # Apply move
    state, reward, done, info = env.step(move)
    
    if done:
        print("\nGame ended early (test complete)")
        break

print("\n✓ Local testing complete!")
print("  Agent appears to be working correctly.")

### Step 5: Deploy to OGS

**Using the deployment script**:

```bash
# Basic deployment (9x9 only)
python scripts/deploy_ogs_bot.py \
    --model models/go_9x9.h5 \
    --board-sizes 9 \
    --mcts \
    --time-control blitz

# Advanced deployment (multiple board sizes)
python scripts/deploy_ogs_bot.py \
    --model models/go_9x9.h5 \
    --board-sizes 9 13 \
    --mcts \
    --mcts-sims 800 \
    --time-control blitz correspondence \
    --max-games 50

# With auto-accept
python scripts/deploy_ogs_bot.py \
    --model models/go_9x9.h5 \
    --board-sizes 9 \
    --auto-accept \
    --min-rank 15k \
    --max-rank 5k
```

**Script Options**:
- `--model`: Path to your trained model (.h5 file)
- `--board-sizes`: Board sizes to accept (9, 13, 19)
- `--mcts`: Enable MCTS (highly recommended)
- `--mcts-sims`: Number of MCTS simulations (default: 400)
- `--time-control`: Accept games with these time controls
- `--auto-accept`: Automatically accept challenges
- `--min-rank` / `--max-rank`: Accept only certain rank ranges
- `--max-games`: Maximum concurrent games

### Step 6: Monitor Your Bot

Once deployed, you can monitor your bot's performance.

In [ ]:
# Simulated bot statistics (in practice, fetch from OGS API)
bot_stats = {
    'games_played': 47,
    'wins': 32,
    'losses': 15,
    'win_rate': 68.1,
    'current_rank': '12k',
    'peak_rank': '10k',
    'average_game_length': 87,  # moves
    'total_playtime': 12.5,  # hours
    'last_game': '2024-11-28 14:32:00'
}

# Display statistics
print("="*70)
print("BOT STATISTICS")
print("="*70)
print(f"\nGames Played: {bot_stats['games_played']}")
print(f"Record: {bot_stats['wins']}W - {bot_stats['losses']}L")
print(f"Win Rate: {bot_stats['win_rate']}%")
print(f"\nCurrent Rank: {bot_stats['current_rank']}")
print(f"Peak Rank: {bot_stats['peak_rank']}")
print(f"\nAverage Game Length: {bot_stats['average_game_length']} moves")
print(f"Total Playtime: {bot_stats['total_playtime']} hours")
print(f"Last Game: {bot_stats['last_game']}")
print("="*70)

# Visualize performance
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Win/Loss pie chart
axes[0].pie(
    [bot_stats['wins'], bot_stats['losses']],
    labels=['Wins', 'Losses'],
    autopct='%1.1f%%',
    colors=['#2ecc71', '#e74c3c'],
    startangle=90
)
axes[0].set_title(f"Win Rate: {bot_stats['win_rate']}%")

# Simulated rank progression
dates = ['Day 1', 'Day 2', 'Day 3', 'Day 4', 'Day 5']
ranks = [18, 15, 13, 11, 10]  # kyu ranks (lower is better)

axes[1].plot(dates, ranks, marker='o', linewidth=2, markersize=10, color='#3498db')
axes[1].set_ylabel('Rank (kyu)')
axes[1].set_title('Rank Progression')
axes[1].grid(True, alpha=0.3)
axes[1].invert_yaxis()  # Lower rank number is better
axes[1].set_ylim(20, 8)

plt.tight_layout()
plt.show()

print("\n💡 Your bot is improving! Rank went from 18k to 10k in 5 days.")

---

## Part 3: Deploying to Lichess (Chess)

### Step 1: Upgrade Account to BOT

**Process**:

1. **Create Lichess account**: https://lichess.org/signup
2. **Get API token**: 
   - Go to: https://lichess.org/account/oauth/token
   - Click "Create token"
   - Select scopes: `bot:play`, `challenge:read`, `challenge:write`
   - Copy token (save it - can't see again!)
3. **Upgrade to BOT**: 
   - Use Lichess API or bot library
   - **Warning**: This is **irreversible**! Account becomes bot-only.

### Step 2: Set Up Credentials

In [ ]:
# Add Lichess token to .env
print("Setting up Lichess credentials...\n")

env_file = Path('.env')
if env_file.exists():
    with open(env_file, 'r') as f:
        content = f.read()
    
    if 'LICHESS_TOKEN' in content:
        print("✓ Lichess token already in .env")
    else:
        with open(env_file, 'a') as f:
            f.write("\nLICHESS_TOKEN=your_lichess_token_here\n")
        print("✓ Added LICHESS_TOKEN to .env")
else:
    print("⚠️  .env file not found. Create it first (see Part 2).")

print("\n⚠️  Remember to replace 'your_lichess_token_here' with your actual token!")

### Step 3: Prepare Chess Agent

In [ ]:
print("Preparing Chess agent for deployment...\n")

# Load or create chess agent
chess_model_path = Path('models/chess.h5')

if chess_model_path.exists():
    print(f"Loading existing chess model: {chess_model_path}")
    chess_agent = PrometheusChessAgent()
    chess_agent.model = tf.keras.models.load_model(str(chess_model_path))
    print(f"✓ Loaded chess agent with {chess_agent.model.count_params():,} parameters")
else:
    print("No existing chess model found. Creating new agent...")
    chess_agent = (
        ModelBuilder()
        .chess()
        .strength('medium')
        .prometheus()
        .build()
    )
    print(f"✓ Created new chess agent with {chess_agent.model.count_params():,} parameters")
    print("\n⚠️  WARNING: This agent is untrained! Train it first before deploying.")

# Add MCTS
print("\nEnhancing with MCTS...")
chess_agent = add_mcts(chess_agent, num_simulations=400, preset='standard')
print("✓ MCTS enabled (400 simulations per move)")

### Step 4: Deploy to Lichess

**Using the deployment script**:

```bash
# Basic deployment
python scripts/deploy_lichess_bot.py \
    --model models/chess.h5 \
    --mcts \
    --time-control bullet blitz

# Advanced deployment
python scripts/deploy_lichess_bot.py \
    --model models/chess.h5 \
    --mcts \
    --mcts-sims 800 \
    --time-control bullet blitz rapid \
    --variants standard chess960 \
    --auto-accept \
    --max-games 10

# Challenge specific users
python scripts/deploy_lichess_bot.py \
    --model models/chess.h5 \
    --challenge user1 user2 user3 \
    --time-control blitz
```

**Script Options**:
- `--model`: Path to your trained chess model
- `--mcts`: Enable MCTS
- `--mcts-sims`: MCTS simulations per move
- `--time-control`: Accept these time controls (bullet/blitz/rapid/classical)
- `--variants`: Chess variants to accept (standard, chess960, etc.)
- `--auto-accept`: Auto-accept all challenges
- `--challenge`: Challenge specific users
- `--max-games`: Maximum concurrent games

---

## Part 4: Managing Multiple Bots

### Running Multiple Bots Simultaneously

You can run multiple bots for different scenarios:

**Example Configuration**:

```yaml
# config/multi_bot.yaml
bots:
  - name: PrometheusGo9x9
    game: go
    model: models/go_9x9.h5
    board_size: 9
    mcts: true
    mcts_sims: 400
    
  - name: PrometheusGo19x19
    game: go
    model: models/go_19x19.h5
    board_size: 19
    mcts: true
    mcts_sims: 800
    
  - name: PrometheusChess
    game: chess
    model: models/chess.h5
    mcts: true
    mcts_sims: 400
```

**Running**:

```bash
# Use process manager (systemd, supervisor, or tmux)
tmux new -s go9x9 'python scripts/deploy_ogs_bot.py --model models/go_9x9.h5 --board-sizes 9'
tmux new -s go19x19 'python scripts/deploy_ogs_bot.py --model models/go_19x19.h5 --board-sizes 19'
tmux new -s chess 'python scripts/deploy_lichess_bot.py --model models/chess.h5'
```

### Bot Management Dashboard

Create a simple monitoring dashboard for all your bots.

In [ ]:
# Simulated multi-bot dashboard
bots_status = [
    {
        'name': 'PrometheusGo9x9',
        'platform': 'OGS',
        'status': 'online',
        'games_today': 12,
        'active_games': 3,
        'win_rate': 67.5,
        'uptime': 23.5  # hours
    },
    {
        'name': 'PrometheusGo19x19',
        'platform': 'OGS',
        'status': 'online',
        'games_today': 5,
        'active_games': 1,
        'win_rate': 54.2,
        'uptime': 23.5
    },
    {
        'name': 'PrometheusChess',
        'platform': 'Lichess',
        'status': 'offline',
        'games_today': 0,
        'active_games': 0,
        'win_rate': 0.0,
        'uptime': 0.0
    }
]

# Display dashboard
print("="*90)
print("BOT MANAGEMENT DASHBOARD")
print("="*90)
print(f"{'Bot Name':<20} {'Platform':<10} {'Status':<10} {'Games':<8} {'Active':<8} {'Win%':<8} {'Uptime':<10}")
print("-"*90)

for bot in bots_status:
    status_icon = "🟢" if bot['status'] == 'online' else "🔴"
    print(
        f"{bot['name']:<20} "
        f"{bot['platform']:<10} "
        f"{status_icon} {bot['status']:<7} "
        f"{bot['games_today']:<8} "
        f"{bot['active_games']:<8} "
        f"{bot['win_rate']:<7.1f}% "
        f"{bot['uptime']:.1f}h"
    )

print("="*90)

# Summary
total_games = sum(b['games_today'] for b in bots_status)
avg_win_rate = np.mean([b['win_rate'] for b in bots_status if b['games_today'] > 0])
online_bots = sum(1 for b in bots_status if b['status'] == 'online')

print(f"\nSummary:")
print(f"  Online bots: {online_bots}/{len(bots_status)}")
print(f"  Total games today: {total_games}")
print(f"  Average win rate: {avg_win_rate:.1f}%")

# Visualize
fig, ax = plt.subplots(figsize=(10, 6))

bot_names = [b['name'] for b in bots_status]
win_rates = [b['win_rate'] for b in bots_status]
colors = ['#2ecc71' if b['status'] == 'online' else '#95a5a6' for b in bots_status]

ax.barh(bot_names, win_rates, color=colors)
ax.set_xlabel('Win Rate (%)')
ax.set_title('Bot Performance Overview')
ax.set_xlim(0, 100)

for i, (name, wr) in enumerate(zip(bot_names, win_rates)):
    if wr > 0:
        ax.text(wr + 2, i, f"{wr:.1f}%", va='center', fontweight='bold')

plt.tight_layout()
plt.show()

---

## Part 5: Monitoring and Debugging

### Common Issues and Solutions

#### Issue 1: Bot Making Illegal Moves

**Symptoms**:
- Games end immediately with forfeit
- Opponent reports illegal moves
- Log shows move validation errors

**Solutions**:
1. **Test locally first** (see Part 2, Step 4)
2. **Check move filtering**: Ensure policy output filters illegal moves
3. **Verify environment**: Make sure board state is parsed correctly
4. **Add validation**: Double-check all moves before sending

```python
# Add move validation
def get_valid_move(agent, state, legal_moves):
    for attempt in range(10):  # Try up to 10 times
        move = agent.get_move(state)
        if move in legal_moves:
            return move
    # Fallback: random legal move
    return np.random.choice(legal_moves)
```

#### Issue 2: Bot Timing Out

**Symptoms**:
- Forfeits due to time
- Very slow move generation
- CPU at 100% constantly

**Solutions**:
1. **Reduce MCTS simulations**: 800 → 400 → 200
2. **Use GPU if available**: Much faster inference
3. **Add time management**: Scale MCTS based on remaining time
4. **Optimize model**: Use quantization (see Part 6)

```python
# Dynamic MCTS based on time
def adaptive_mcts_sims(time_remaining, time_per_move):
    if time_remaining < 30:  # Less than 30 seconds
        return 100
    elif time_remaining < 60:
        return 200
    else:
        return 400
```

#### Issue 3: Bot Disconnecting

**Symptoms**:
- Bot goes offline randomly
- Connection errors in logs
- Games abandoned

**Solutions**:
1. **Add reconnection logic**: Auto-reconnect on disconnect
2. **Use process manager**: systemd, supervisor, or Docker
3. **Monitor health**: Ping server periodically
4. **Check network**: Ensure stable internet connection

```bash
# Use systemd for auto-restart
# /etc/systemd/system/prometheus-go-bot.service
[Unit]
Description=Prometheus Go Bot
After=network.target

[Service]
Type=simple
User=youruser
WorkingDirectory=/path/to/Prometheus_v0_PoC
ExecStart=/usr/bin/python3 scripts/deploy_ogs_bot.py --model models/go_9x9.h5
Restart=always
RestartSec=10

[Install]
WantedBy=multi-user.target
```

### Logging and Monitoring

**Use Prometheus logging system**:

In [ ]:
from prometheus.utils import get_logger, setup_logging

# Setup logging
setup_logging(
    level='INFO',
    log_file='logs/bot_deployment.log',
    console=True
)

logger = get_logger(__name__)

# Log bot events
logger.info("Bot started successfully")
logger.info("Connected to OGS")
logger.info("Accepted challenge from user123")
logger.warning("Move generation took 5.2s (slow!)")
logger.error("Failed to connect to server", exc_info=True)

print("\n✓ Logging configured")
print("  Console: Enabled")
print("  File: logs/bot_deployment.log")

---

## Part 6: Performance Optimization

### Model Quantization

**Reduce model size and inference time by 2-4x**:

In [ ]:
print("Model Quantization Example\n")

# Load model
if Path('models/go_9x9.h5').exists():
    model = tf.keras.models.load_model('models/go_9x9.h5')
    
    # Original model size
    original_size = Path('models/go_9x9.h5').stat().st_size / 1024 / 1024
    print(f"Original model size: {original_size:.2f} MB")
    
    # Quantize to INT8
    print("\nQuantizing to INT8...")
    
    # Convert to TFLite with quantization
    converter = tf.lite.TFLiteConverter.from_keras_model(model)
    converter.optimizations = [tf.lite.Optimize.DEFAULT]
    quantized_model = converter.convert()
    
    # Save quantized model
    quantized_path = Path('models/go_9x9_quantized.tflite')
    quantized_path.write_bytes(quantized_model)
    
    quantized_size = quantized_path.stat().st_size / 1024 / 1024
    print(f"✓ Quantized model size: {quantized_size:.2f} MB")
    print(f"  Size reduction: {100 * (1 - quantized_size/original_size):.1f}%")
    print(f"  Saved: {original_size - quantized_size:.2f} MB")
    
    print("\n💡 Quantized models are 2-4x faster with minimal accuracy loss!")
else:
    print("No model found. Skipping quantization demo.")

### Position Caching

**Cache MCTS evaluations for repeated positions**:

In [ ]:
# Position cache example
import hashlib
from functools import lru_cache

class PositionCache:
    """Cache MCTS evaluations for repeated positions."""
    
    def __init__(self, max_size=10000):
        self.cache = {}
        self.max_size = max_size
        self.hits = 0
        self.misses = 0
    
    def get_hash(self, state):
        """Get hash of board state."""
        return hashlib.md5(state.tobytes()).hexdigest()
    
    def get(self, state):
        """Get cached evaluation if exists."""
        key = self.get_hash(state)
        if key in self.cache:
            self.hits += 1
            return self.cache[key]
        self.misses += 1
        return None
    
    def put(self, state, value):
        """Cache evaluation."""
        if len(self.cache) >= self.max_size:
            # Remove oldest entry (simple FIFO)
            self.cache.pop(next(iter(self.cache)))
        key = self.get_hash(state)
        self.cache[key] = value
    
    def hit_rate(self):
        """Calculate cache hit rate."""
        total = self.hits + self.misses
        return 100 * self.hits / total if total > 0 else 0

# Demo
cache = PositionCache(max_size=1000)

# Simulate position lookups
state1 = np.random.rand(9, 9, 3)
state2 = np.random.rand(9, 9, 3)

# First lookup (miss)
result = cache.get(state1)
if result is None:
    cache.put(state1, {'policy': [0.1] * 81, 'value': 0.5})

# Second lookup (hit)
result = cache.get(state1)

# Different state (miss)
result = cache.get(state2)
if result is None:
    cache.put(state2, {'policy': [0.1] * 81, 'value': 0.3})

# Third lookup of state1 (hit)
result = cache.get(state1)

print(f"Cache statistics:")
print(f"  Hits: {cache.hits}")
print(f"  Misses: {cache.misses}")
print(f"  Hit rate: {cache.hit_rate():.1f}%")
print(f"\n💡 Cache can give 2-5x speedup in MCTS!")

---

## Part 7: Best Practices

### ✅ DO:

1. **Test locally before deploying**
   - Verify agent makes legal moves
   - Check performance on test games
   - Measure move generation time

2. **Start small**
   - Deploy one bot first
   - Monitor for a day
   - Scale up gradually

3. **Use appropriate time controls**
   - Blitz/Rapid for fast bots (MCTS ≤400 sims)
   - Correspondence for strong bots (MCTS 800+)
   - Avoid bullet unless bot is very fast

4. **Monitor continuously**
   - Check logs daily
   - Track win rate and rank
   - Watch for errors

5. **Be a good citizen**
   - Don't spam challenges
   - Respect time controls
   - Label bot accounts clearly
   - Follow platform rules

### ❌ DON'T:

1. **Don't deploy untested bots**
   - Will make illegal moves
   - Wastes opponents' time
   - May get banned

2. **Don't run too many games simultaneously**
   - Will timeout
   - Lower quality play
   - May crash

3. **Don't ignore errors**
   - Check logs regularly
   - Fix issues promptly
   - Don't let bot run broken

4. **Don't use same account for humans and bots**
   - Against platform rules
   - May get banned
   - Creates unfair advantages

5. **Don't forget rate limits**
   - Respect API rate limits
   - Don't spam the server
   - Use exponential backoff

### Deployment Checklist

Before deploying, ensure:

- [ ] Agent is trained and tested locally
- [ ] Bot account created and approved
- [ ] API tokens configured in .env
- [ ] Deployment script tested
- [ ] Logging configured
- [ ] Monitoring dashboard ready
- [ ] Process manager configured (systemd/supervisor)
- [ ] Backup strategy in place
- [ ] Error handling tested
- [ ] Time management configured
- [ ] Platform rules reviewed

---

## Summary

### What We Learned

1. **OGS Deployment (Go)**
   - Create bot account and get API token
   - Use deployment script for easy setup
   - Monitor performance and adjust

2. **Lichess Deployment (Chess)**
   - Upgrade account to BOT (irreversible!)
   - Configure API token
   - Deploy with appropriate settings

3. **Multi-Bot Management**
   - Run multiple bots for different games/sizes
   - Use process managers for reliability
   - Monitor all bots from dashboard

4. **Monitoring & Debugging**
   - Common issues and solutions
   - Logging best practices
   - Performance optimization

5. **Optimization**
   - Model quantization (2-4x faster)
   - Position caching (2-5x MCTS speedup)
   - Time management strategies

### Quick Reference

**Deploy Go bot to OGS**:
```bash
python scripts/deploy_ogs_bot.py --model models/go_9x9.h5 --mcts
```

**Deploy Chess bot to Lichess**:
```bash
python scripts/deploy_lichess_bot.py --model models/chess.h5 --mcts
```

**Monitor logs**:
```bash
tail -f logs/bot_deployment.log
```

**Restart bot (systemd)**:
```bash
sudo systemctl restart prometheus-go-bot
```

### Next Steps

1. **Deploy your first bot**: Start with OGS (easiest)
2. **Monitor for 24 hours**: Check for errors and performance
3. **Optimize if needed**: Reduce MCTS sims if too slow
4. **Scale up**: Add more bots for different scenarios
5. **Iterate**: Retrain based on real game data

---

**Congratulations!** You now know how to deploy Prometheus agents to production! 🚀